<a href="https://colab.research.google.com/github/FRakhmatov/desktop-tutorial/blob/main/Real-Time_Zero-Day_Attack_Detection_Using_a_Lightweight_Cascade_Architecture_test_in_UNSW-NB15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div class="markdown-google-sans">

<a name="machine-learning-examples"></a>

### Real-Time Zero-Day Attack Detection Using a Lightweight Cascade Architecture
Dataset: UNSW-NB15

</div>


# Real-Time Zero-Day Attack Detection Using a Lightweight Cascade Architecture

This project implements a real-time zero-day attack detection system using a lightweight cascade architecture, designed to efficiently identify malicious network traffic. The system leverages a multi-stage approach, combining two XGBoost classifiers, to balance recall and precision in detecting sophisticated attacks.

## Dataset

The model is trained and evaluated using the **UNSW-NB15 dataset**, a well-known benchmark for network intrusion detection.

## Features

*   **Zero-Day Split**: The dataset is split into training and testing sets such that attack types present in the test set are entirely unseen during training, simulating a zero-day scenario.
*   **Two-Stage XGBoost Classification**:
    *   **Stage 1 (Recall-first)**: A highly sensitive XGBoost model designed to catch as many potential attacks as possible.
    *   **Stage 2 (Precision-first)**: A more precise XGBoost model that re-evaluates suspicious samples flagged by Stage 1, reducing false positives.
*   **Performance Metrics**: Provides accuracy, False Positive Rate (FPR), latency, throughput, and confusion matrix.
*   **ROC/PR Curves**: Visualizations for detailed performance analysis.

## How it Works

### Architecture Overview

The system employs a cascade architecture:

1.  **Data Loading and Splitting**: The `load_unsw_zero_day` function loads the UNSW-NB15 dataset and splits it into training and testing sets based on predefined attack categories (`TRAIN_ATTACKS` and `TEST_ATTACKS`), ensuring that certain attack types are only present in the test set (zero-day scenario).
2.  **Feature Scaling**: Numerical features are scaled using `StandardScaler` to normalize their ranges.
3.  **Stage 1 – Recall-first XGBoost Classifier**: This model is trained on the scaled training data, with `scale_pos_weight` adjusted for class imbalance. It aims to identify the vast majority of malicious samples, even if it produces a higher number of false positives. Its predictions (`scores1_test`) are used to identify `suspicious` samples that score above a threshold (`TH1`).
4.  **Stage 2 – Precision-first XGBoost Classifier**: This model is trained only on the subset of training data that was flagged as suspicious by Stage 1. It is optimized for high precision, carefully distinguishing true attacks from benign but unusual samples, thus reducing the overall False Positive Rate. It then predicts on the `suspicious` subset of the test data.
5.  **Final Decision & Scores**: The final prediction (`y_pred`) and scores (`final_scores`) are determined by combining the results from Stage 1 and Stage 2.

## Installation

This project requires the following Python libraries:

```bash
pip install pandas numpy scikit-learn xgboost matplotlib psutil
```

## Usage

1.  **Prepare your data**: Ensure your UNSW-NB15 dataset is available as a CSV file. The script assumes the file is located at `/content/sample_data/UNSW_NB15.csv`.
2.  **Define Attack Categories**: Adjust `TRAIN_ATTACKS` and `TEST_ATTACKS` lists in the `if __name__ == "__main__":` block according to your desired zero-day split.
3.  **Run the script**: Execute the Python script. It will load the data, train both stages of the cascade model, and evaluate its performance.

```python
# Example usage block from the script
if __name__ == "__main__":

    DATA_PATH = "/content/sample_data/UNSW_NB15.csv"

    TRAIN_ATTACKS = ["Fuzzers", "DoS", "Reconnaissance"]
    TEST_ATTACKS  = ["Exploits", "Backdoor", "Shellcode", "Worms"]

    TH1 = 0.15 # Threshold for Stage 1 (Recall-first)
    TH2 = 0.85 # Threshold for Stage 2 (Precision-first)

    print("🔄 Loading UNSW-NB15 with zero-day split...")
    X_train_raw, y_train, X_test_raw, y_test = load_unsw_zero_day(
        DATA_PATH,
        TRAIN_ATTACKS,
        TEST_ATTACKS
    )

    # ... (rest of the training and evaluation code)
```

## Evaluation Results

After training and evaluation on the UNSW-NB15 dataset with the specified zero-day split, the system produced the following results:

```
🔄 Loading UNSW-NB15 with zero-day split...
🚀 Training Stage-1 (Recall-first)...
🚀 Training Stage-2 (Precision-first)...

📊 UNSW ZERO-DAY CASCADE RESULTS
Accuracy     : 0.9976
FPR          : 0.0042
Latency      : 0.0066 ms
Throughput   : 152492 req/s
Memory (MB)  : 0.00
Confusion Matrix:
[[1675    7]
 [  11 5936]]
```

### Interpretation of Results:

*   **Accuracy**: 99.76% of all samples were correctly classified.
*   **FPR (False Positive Rate)**: Only 0.42% of benign samples were incorrectly flagged as malicious, indicating a very low rate of false alarms.
*   **Latency**: Extremely low inference time per sample (0.0066 ms), suitable for real-time applications.
*   **Throughput**: High processing capacity, handling approximately 152,492 requests per second.
*   **Memory (MB)**: 0.00 MB indicates very efficient memory usage during the measurement of inference. (Note: This likely refers to the *change* in memory during inference, not total memory consumption).
*   **Confusion Matrix**:
    *   `1675`: True Negatives (correctly identified benign samples).
    *   `7`: False Positives (benign samples incorrectly flagged as malicious).
    *   `11`: False Negatives (malicious samples missed by the system).
    *   `5936`: True Positives (correctly identified malicious samples).

These results demonstrate the system's effectiveness in detecting zero-day attacks on the UNSW-NB15 dataset with very high accuracy, low false positive rate, and impressive real-time performance capabilities.

In [ ]:
import pandas as pd
import numpy as np
import time
import psutil
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    auc
)
from xgboost import XGBClassifier


# =====================================================
# Utilities
# =====================================================

def memory_mb():
    return psutil.Process().memory_info().rss / (1024 ** 2)


# =====================================================
# Load UNSW-NB15 with attack-type zero-day split
# =====================================================

def load_unsw_zero_day(path, train_attack_types, test_attack_types):
    df = pd.read_csv(path)

    # Drop rows where 'label' is NaN before converting to int
    df.dropna(subset=["label"], inplace=True)
    df["label"] = df["label"].astype(int)

    is_train = (df["label"] == 0) | (df["attack_cat"].isin(train_attack_types))
    is_test  = (df["label"] == 0) | (df["attack_cat"].isin(test_attack_types))

    df_train = df[is_train].copy()
    df_test  = df[is_test].copy()

    drop_cols = ["id", "proto", "service", "state", "attack_cat"]
    df_train.drop(columns=drop_cols, inplace=True)
    df_test.drop(columns=drop_cols, inplace=True)

    X_train = df_train.drop(columns=["label"])
    y_train = df_train["label"].values

    X_test = df_test.drop(columns=["label"])
    y_test = df_test["label"].values

    return X_train, y_train, X_test, y_test


# =====================================================
# Plot ROC & PR curves (publication quality)
# =====================================================

def plot_roc_pr(y_true, y_scores, title):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    pr_auc = auc(recall, precision)

    plt.figure(figsize=(12, 5))

    # ROC
    plt.subplot(1, 2, 1)
    plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], "--", linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title} ROC Curve")
    plt.legend()
    plt.grid(alpha=0.3)

    # PR
    plt.subplot(1, 2, 2)
    plt.plot(recall, precision, linewidth=2, label=f"AUC = {pr_auc:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title} Precision–Recall Curve")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# =====================================================
# MAIN
# =====================================================

if __name__ == "__main__":

    DATA_PATH = "/content/sample_data/UNSW_NB15.csv"  # объединённый файл

    TRAIN_ATTACKS = ["Fuzzers", "DoS", "Reconnaissance"]
    TEST_ATTACKS  = ["Exploits", "Backdoor", "Shellcode", "Worms"]

    TH1 = 0.15
    TH2 = 0.85

    print("🔄 Loading UNSW-NB15 with zero-day split...")
    X_train_raw, y_train, X_test_raw, y_test = load_unsw_zero_day(
        DATA_PATH,
        TRAIN_ATTACKS,
        TEST_ATTACKS
    )

    # ===============================
    # Scaling
    # ===============================
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test  = scaler.transform(X_test_raw)

    neg, pos = np.bincount(y_train)

    # ===============================
    # Stage-1 (Recall-first)
    # ===============================
    stage1 = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        scale_pos_weight=neg / pos,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    )

    print("🚀 Training Stage-1 (Recall-first)...")
    stage1.fit(X_train, y_train)

    scores1_test = stage1.predict_proba(X_test)[:, 1]
    suspicious = scores1_test > TH1

    # ===============================
    # Stage-2 (Precision-first)
    # ===============================
    stage2 = XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        gamma=1.0,
        reg_alpha=0.5,
        reg_lambda=1.0,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    )

    scores1_train = stage1.predict_proba(X_train)[:, 1]
    train_mask = scores1_train > TH1

    if len(np.unique(y_train[train_mask])) < 2:
        print("⚠️ Stage-2 skipped (single class after filtering)")
        stage2 = None
    else:
        print("🚀 Training Stage-2 (Precision-first)...")
        stage2.fit(X_train[train_mask], y_train[train_mask])

    # ===============================
    # Final decision & scores
    # ===============================
    y_pred = np.zeros_like(y_test)
    final_scores = np.zeros_like(y_test, dtype=float)

    if stage2 is not None:
        scores2 = stage2.predict_proba(X_test[suspicious])[:, 1]
        y_pred[suspicious] = (scores2 > TH2).astype(int)
        final_scores[suspicious] = scores2
    else:
        y_pred[suspicious] = 1
        final_scores[suspicious] = scores1_test[suspicious]

    final_scores[~suspicious] = scores1_test[~suspicious] * 0.3

    # ===============================
    # Performance metrics
    # ===============================
    mem_before = memory_mb()
    t0 = time.time()
    _ = stage1.predict(X_test)
    infer_time = (time.time() - t0) * 1000
    mem_after = memory_mb()

    latency = infer_time / X_test.shape[0]
    throughput = 1000 / latency

    cm = confusion_matrix(y_test, y_pred)
    accuracy = np.mean(y_test == y_pred)
    fpr = cm[0, 1] / (cm[0, 0] + cm[0, 1])

    print("\n📊 UNSW ZERO-DAY CASCADE RESULTS")
    print(f"Accuracy     : {accuracy:.4f}")
    print(f"FPR          : {fpr:.4f}")
    print(f"Latency      : {latency:.4f} ms")
    print(f"Throughput   : {throughput:.0f} req/s")
    print(f"Memory (MB)  : {mem_after - mem_before:.2f}")
    print("Confusion Matrix:")
    print(cm)

    # ===============================
    # ROC / PR Curves
    # ===============================
    plot_roc_pr(
        y_true=y_test,
        y_scores=final_scores,
        title="UNSW-NB15 Zero-Day"
    )

    print("\n✅ Zero-day UNSW experiment completed successfully.")